## Problem 1: Representing SHA-256 Data

SHA-256 is specified in [NIST FIPS 180-4](https://csrc.nist.gov/files/pubs/fips/180-4/final/docs/fips180-4.pdf), which describes how messages of less than 2<sup>64</sup> bits are processed to produce a 256-bit message digest. SHA-256 uses fixed-width 32-bit words, whereas Python's built-in `int` type can represent integers of arbitrary size. The [Python documentation](https://docs.python.org/3/library/stdtypes.html#bitwise-operations-on-integer-types) explains that the bitwise AND operator `&` can be used with the mask `0xFFFFFFFF` to retain the lowest 32 bits of an integer. This section examines how Python data types can represent the input, output, and intermediate data structures used in SHA-256, without implementing the hashing operations or message padding.

### 1. 32-bit words

SHA-256 operates on 32-bit words, which can represent unsigned values from 0 to 2<sup>32</sup>-1 (Max unsigned int value). In Python, these values can be represented using the built-in `int` type and written in hexadecimal for readability. For example, `0x6A09E667` is one of the initial hash values specified for SHA-256 in NIST FIPS 180-4.

Python integers can store values larger than 32 bits, so the programmer must ensure that values remain within the required range when necessary. The bitwise AND operator `&` can be used with `0xFFFFFFFF` as a mask. This retains the lowest 32 bits and removes any bits above them.
.

In [1]:
word_32 = 0x6A09E667 # From section 5.3.3 in NIST docs

print(f"SHA-256 word: 0x{word_32:08x}")
print(f"Word size: {word_32.bit_length()} bits")

SHA-256 word: 0x6a09e667
Word size: 31 bits


#### Output Explained

The output shows the word size is 31 bits. This is because `6` in binary is `0110` and the initial `0` is ignored so python counts 31 bits.

### 2. Sequences of 32 bit words

NIST FIPS 180-4 specifies that SHA-256 uses 32-bit words. For example, each 512-bit message block is represented as sixteen 32-bit words, and the message schedule contains 64 words during the hash computation. These sequences can be represented in Python using a list[int]. According to the Python documentation, lists are ordered, mutable sequences, making them suitable for storing and updating word values during computation. Since Python integers are not restricted to 32 bits, each element can be checked or masked to ensure that it remains within the required 32-bit range. A tuple could be used when the sequence should remain unchanged.

#### References
NIST FIPS 180-4, Sections 2.1, 5.2.1 and 6.2.2; Python documentation, [Built-in Types — Lists and Tuples](https://docs.python.org/3/builtins/stdtypes.html#sequence-types-list-tuple-range).
.

In [2]:
# The eight initial hash values specified for SHA-256 in NIST FIPS 180-4.
initial_hash_values = [
    0x6A09E667,
    0xBB67AE85,
    0x3C6EF372,
    0xA54FF53A,
    0x510E527F,
    0x9B05688C,
    0x1F83D9AB,
    0x5BE0CD19,
]

# Check the number of words and their permitted range.
assert len(initial_hash_values) == 8
assert all(0 <= word <= 0xFFFFFFFF for word in initial_hash_values)

for index, word in enumerate(initial_hash_values):
    print(f"H[{index}]: 0x{word:08x}")

H[0]: 0x6a09e667
H[1]: 0xbb67ae85
H[2]: 0x3c6ef372
H[3]: 0xa54ff53a
H[4]: 0x510e527f
H[5]: 0x9b05688c
H[6]: 0x1f83d9ab
H[7]: 0x5be0cd19


#### Output Explained
This code snippet was taken from [NIST 6.2.2](https://csrc.nist.gov/files/pubs/fips/180-4/final/docs/fips180-4.pdf). Whats being demonstrated is;
- list[int]: stores the eight words in a defined order.
- Hexadecimal values: match the notation used in the NIST specification.
- len(): confirms that the sequence contains eight words.
- assert: checks that every integer fits within the unsigned 32-bit range.
- enumerate(): displays each word alongside its index.

### Problem 1.3: Input Messages

According to section 5.1.1 of NIST FIPS 180-4, SHA-256 processes an input message represented as a bit string of length <var>l</var>, where 0 &le; <var>l</var> &lt; 2<sup>64</sup> bits. Section 6.2 specifies the SHA-256 algorithm itself.

In Python, byte-oriented message data can be represented using the built-in `bytes` type. Each byte consists of eight bits. Although the NIST standard defines messages as bit strings, a Python `bytes` object represents a sequence of complete bytes.

#### Suitable Python Data Types

- **`bytes`**: An immutable sequence of byte values from 0 to 255. This is a suitable representation for binary SHA-256 input.
- **`bytearray`**: A mutable sequence of byte values, useful when constructing or modifying message data before hashing.
- **Text strings (`str`) versus byte sequences**: Python strings represent Unicode text, whereas `bytes` represents binary data. Text must therefore be encoded before it is passed to a byte-oriented SHA-256 implementation. UTF-8 is a common encoding.

#### Message Bit-Length Calculation

According to **Section 5.1.1 of NIST FIPS 180-4**, SHA-256 preprocessing appends padding and incorporates the original message length \(l\), measured in bits, into the padded message.

For a Python byte sequence, the original message length in bits is calculated as:

<var>l</var> = 8 &times; <var>len</var>(<var>M</var>)

where \(M\) is the original byte sequence. This calculation must be performed before padding is applied. The original message length is distinct from the length of the padded message.


In [3]:

# Representing input messages in Python

# 1. Represent text as an immutable bytes object
message = "abc"
msg_bytes = message.encode("utf-8")

print("Encoded message:", msg_bytes)
print("Data type:", type(msg_bytes))
print("Byte values:", list(msg_bytes))

# 2. Represent binary input directly as bytes
binary_message = b"abc"
print("Direct bytes:", binary_message)

# 3. Use bytearray when the message needs modification
mutable_message = bytearray(b"abc")
mutable_message.append(100)  # Append the byte value for 'd'
print("Modified bytearray:", mutable_message)

# Convert to immutable bytes when ready
final_message = bytes(mutable_message)
print("Final bytes:", final_message)


Encoded message: b'abc'
Data type: <class 'bytes'>
Byte values: [97, 98, 99]
Direct bytes: b'abc'
Modified bytearray: bytearray(b'abcd')
Final bytes: b'abcd'


In [4]:

# Calculating the original message length in bits
# Based on the "abc" example in NIST FIPS 180-4, Section 5.1.1

msg = b"abc"

message_length_bytes = len(msg)
message_length_bits = message_length_bytes * 8

print("Message:", msg)
print("Length in bytes:", message_length_bytes)
print("Length in bits:", message_length_bits)


Message: b'abc'
Length in bytes: 3
Length in bits: 24


### Problem 1.4 512-bit message blocks

NIST FIPS 180-4, Section 5.2.1, specifies that SHA-256 messages and their padding are divided into 512-bit blocks, with each block represented as sixteen 32-bit words. This gives a total of 512 bits (16 × 32). In Python, a tuple provides an ordered, immutable sequence, as explained in the official [Python documentation on tuples](https://docs.python.org/3/builtins/stdtypes.html#tuples). This makes a tuple suitable for representing a block whose word entries should not be reassigned after creation. However, a tuple does not enforce the number of entries or the size of each integer, so assertions are used to check that the block contains sixteen words and that every word fits within the unsigned 32-bit range. The example illustrates a block layout for the message abc; it does not implement the padding procedure.

In [1]:
WORD_MASK = 0xFFFFFFFF

# Illustrative 512-bit block layout for the message "abc".
message_block = (
    0x61626380, 0x00000000, 0x00000000, 0x00000000,
    0x00000000, 0x00000000, 0x00000000, 0x00000000,
    0x00000000, 0x00000000, 0x00000000, 0x00000000,
    0x00000000, 0x00000000, 0x00000000, 0x00000018,
)

assert len(message_block) == 16
assert all(0 <= word <= WORD_MASK for word in message_block)

block_size_bits = len(message_block) * 32
assert block_size_bits == 512

print(f"Words in block: {len(message_block)}")
print(f"Block size: {block_size_bits} bits")
print(f"First word: 0x{message_block[0]:08x}")
print(f"Last word: 0x{message_block[-1]:08x}")

Words in block: 16
Block size: 512 bits
First word: 0x61626380
Last word: 0x00000018


### Problem 1.5 The Final 256-Bit Hash Value

SHA-256 produces a 256-bit message digest, as specified in NIST FIPS 180-4, Section 6.2. The final digest consists of eight 32-bit words, which are combined to form the 256-bit hash value. This is equivalent to 32 bytes, since each byte contains 8 bits.

In Python, the digest can be represented as a bytes object using the [hashlib digest() method](https://docs.python.org/3/library/hashlib.html#hash.digest). To display the digest in hexadecimal form, the [bytes.hex() method](https://docs.python.org/3/builtins/stdtypes.html#bytes.hex) converts each byte into two hexadecimal characters. Consequently, the 256-bit digest is represented by a string of 64 hexadecimal characters.

The following example uses the [known SHA-256 digest of the message abc](https://www.nist.gov/itl/ai/ai-standards-and-guidelines-group/nsrl-test-data?utm_source=chatgpt.com) to demonstrate the byte and hexadecimal representations.

In [1]:

import hashlib

# Calculate the SHA-256 digest of the message "abc".
message = b"abc"
digest = hashlib.sha256(message).digest()

# Convert the digest to hexadecimal.
hex_digest = digest.hex()

# Verify the digest representation.
assert len(digest) == 32
assert len(digest) * 8 == 256
assert len(hex_digest) == 64

print(f"Digest: {digest}")
print(f"Digest size: {len(digest)} bytes")
print(f"Digest size: {len(digest) * 8} bits")
print(f"Hexadecimal digest: {hex_digest}")


Digest: b'\xbax\x16\xbf\x8f\x01\xcf\xeaAA@\xde]\xae"#\xb0\x03a\xa3\x96\x17z\x9c\xb4\x10\xffa\xf2\x00\x15\xad'
Digest size: 32 bytes
Digest size: 256 bits
Hexadecimal digest: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


#### Output Explained
- The first output results in  `b'\xbax\x16\xbf\x8f\x01\xcf\xeaAA@\xde]\xae"#\xb0\x03a\xa3\x96\x17z\x9c\xb4\x10\xffa\xf2\x00\x15\xad'` this is the raw bytes. Some bytes are not displayable as printable text so python uses escape sequences to print them; `\xba`, `x16` and `xbf` are examples of this.
- The second output shows 32 bytes, even though when its printed the escpae sequences make it look much longer.
- The third output its the multiplication of 32 x 8 to get 256 bits.
- The fourth output is the actual 256 has value in 64 hexadecimal form. It is equilivant to 32 byts.

